In [2]:
import os
import pickle
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

In [3]:
import os
from pathlib import Path

# Root of the project. Defaults to the current working directory,
# or override via the RAG_BASE_DIR environment variable.
BASE_DIR = Path(os.getenv("RAG_BASE_DIR", Path.cwd()))
VECTOR_DIR = BASE_DIR / "VectorDB"

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

In [4]:
faiss_index = faiss.read_index(os.path.join(VECTOR_DIR, "faiss.index"))

with open(os.path.join(VECTOR_DIR, "texts.pkl"), "rb") as f:
    ALL_TEXTS = pickle.load(f)

with open(os.path.join(VECTOR_DIR, "metadatas.pkl"), "rb") as f:
    ALL_METADATAS = pickle.load(f)

assert faiss_index.ntotal == len(ALL_TEXTS) == len(ALL_METADATAS)

RuntimeError: Error in __cdecl faiss::FileIOReader::FileIOReader(const char *) at D:\a\faiss-wheels\faiss-wheels\third-party\faiss\faiss\impl\io.cpp:70: Error: 'f' failed: could not open C:\Users\sriva\Desktop\projects\RAG 2.0\VectorDB\faiss.index for reading: No such file or directory

In [4]:
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
reranker = CrossEncoder(RERANKER_MODEL_NAME)

In [5]:
def stage1_retrieve(query: str, top_k: int = 20):
    query_embedding = embed_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = faiss_index.search(query_embedding, top_k)

    results = []
    for idx in indices[0]:
        results.append({
            "text": ALL_TEXTS[idx],
            "metadata": ALL_METADATAS[idx]
        })

    return results

In [ ]:
def stage2_rerank(query: str, candidates: list, top_k: int = 5):
    pairs = [(query, c["text"]) for c in candidates]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [r[0] for r in ranked[:top_k]]

In [8]:
def retrieve_context(query: str, faiss_k: int = 20, final_k: int = 5):
    stage1_results = stage1_retrieve(query, top_k=faiss_k)
    stage2_results = stage2_rerank(query, stage1_results, top_k=final_k)

    context_blocks = []
    sources = []

    for r in stage2_results:
        context_blocks.append(r["text"])
        sources.append(
            f'{r["metadata"]["pdf_name"]}, page {r["metadata"]["page"]}'
        )

    context = "\n\n---\n\n".join(context_blocks)

    return {
        "context": context,
        "sources": sources
    }